# Airline Passenger Satisfaction Analysis & Prediction
## Comprehensive Exploratory Data Analysis, Logistic Regression, and Model Evaluation

Based on the Project Report:
- **Project Area**: Data Analytics and Machine Learning
- **Dataset**: Airline Passenger Satisfaction (Kaggle)
- **Core Algorithm**: Logistic Regression (Scikit-learn)
- **Complete Visualizations**: All 20 EDA charts from Section 8.6 Recommended EDA Chart Inventory, plus complete Model Performance Visualizations (Section 9.5).


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

# Global aesthetic settings
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('All analytical and modeling libraries imported successfully.')


## 1. Load Datasets & Structural Verification
We load `train.csv` and `test.csv` without modifying or deleting original files.


In [ ]:
train_path = 'train.csv' if os.path.exists('train.csv') else os.path.join('..', 'train.csv')
test_path = 'test.csv' if os.path.exists('test.csv') else os.path.join('..', 'test.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()


## 2. Dataset Inspection, Missing Values & Duplicates
We inspect data types, check null counts (specifically `Arrival Delay in Minutes`), and remove record identifier columns (`id`, `Unnamed: 0`).


In [ ]:
print('--- Missing Values in Train ---')
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print('\n--- Missing Values in Test ---')
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

print(f'\nDuplicate rows in Train: {train_df.duplicated().sum()}')
print(f'Duplicate rows in Test:  {test_df.duplicated().sum()}')

# Drop identifier columns if present
drop_cols = [c for c in ['Unnamed: 0', 'id'] if c in train_df.columns]
if drop_cols:
    train_df = train_df.drop(columns=drop_cols)
    test_df = test_df.drop(columns=drop_cols)
    print(f'Dropped non-predictive identifier columns: {drop_cols}')


## 3. Exploratory Data Analysis (EDA) - Complete 20-Chart Inventory
Implementing all visualizations specified in **Section 8.6 (Recommended EDA chart inventory)** of the project report.


### Chart 1: Satisfaction Count & Donut Chart
**Question Answered**: How are target classes distributed?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count Plot
sns.countplot(data=train_df, x='satisfaction', ax=axes[0], palette=['#f43f5e', '#10b981'])
axes[0].set_title('1A. Passenger Satisfaction Count Distribution')
axes[0].set_xlabel('Satisfaction')
axes[0].set_ylabel('Number of Passengers')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 7), textcoords='offset points')

# Donut Chart
sat_counts = train_df['satisfaction'].value_counts()
axes[1].pie(sat_counts, labels=sat_counts.index, autopct='%1.1f%%', startangle=90,
            colors=['#f43f5e', '#10b981'], wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2))
axes[1].set_title('1B. Satisfaction Proportion (Donut Chart)')

plt.tight_layout()
plt.show()


### Chart 2: Gender Distribution
**Question Answered**: What is the gender composition?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=train_df, x='Gender', ax=axes[0], palette=['#3b82f6', '#ec4899'])
axes[0].set_title('2A. Passenger Gender Count')
axes[0].set_ylabel('Count')

gender_counts = train_df['Gender'].value_counts()
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#3b82f6', '#ec4899'], wedgeprops=dict(width=0.4, edgecolor='w'))
axes[1].set_title('2B. Gender Composition')

plt.tight_layout()
plt.show()


### Chart 3: Age Histogram with KDE
**Question Answered**: What age ranges are represented?


In [ ]:
plt.figure(figsize=(10, 5))
mean_age = train_df['Age'].mean()
median_age = train_df['Age'].median()

sns.histplot(data=train_df, x='Age', kde=True, bins=35, color='#2563eb')
plt.title('3. Passenger Age Distribution')
plt.xlabel('Age (Years)')
plt.ylabel('Density / Frequency')
plt.axvline(mean_age, color='red', linestyle='--', label=f'Mean Age: {mean_age:.1f}')
plt.axvline(median_age, color='green', linestyle='-', label=f'Median Age: {median_age:.1f}')
plt.legend()
plt.show()


### Chart 4: Customer Type Distribution
**Question Answered**: How many loyal and disloyal customers are present?


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=train_df, x='Customer Type', palette=['#10b981', '#f59e0b'])
plt.title('4. Customer Type Distribution')
plt.xlabel('Customer Loyalty Tier')
plt.ylabel('Count')
for p in ax.patches:
    pct = 100 * p.get_height() / len(train_df)
    ax.annotate(f'{int(p.get_height()):,} ({pct:.1f}%)', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 6), textcoords='offset points')
plt.show()


### Chart 5: Class Distribution
**Question Answered**: How are passengers distributed by cabin class?


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=train_df, x='Class', palette='Blues_r', order=['Business', 'Eco', 'Eco Plus'])
plt.title('5. Distribution of Passengers by Cabin Class')
plt.xlabel('Travel Class')
plt.ylabel('Count')
for p in ax.patches:
    pct = 100 * p.get_height() / len(train_df)
    ax.annotate(f'{int(p.get_height()):,} ({pct:.1f}%)', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 6), textcoords='offset points')
plt.show()


### Chart 6: Flight Distance Histogram
**Question Answered**: What distances are common?


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=train_df, x='Flight Distance', kde=True, bins=45, color='#0284c7')
plt.title('6. Flight Distance Distribution')
plt.xlabel('Flight Distance (Miles)')
plt.ylabel('Passenger Count')
plt.show()


### Chart 7: Delay Histograms (Departure & Arrival)
**Question Answered**: How are departure and arrival delays distributed?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dep_subset = train_df[train_df['Departure Delay in Minutes'].between(1, 180)]
arr_subset = train_df[train_df['Arrival Delay in Minutes'].between(1, 180)]

sns.histplot(dep_subset['Departure Delay in Minutes'], bins=30, kde=True, ax=axes[0], color='#f59e0b')
axes[0].set_title('7A. Departure Delay Distribution (1 - 180 mins)')
axes[0].set_xlabel('Departure Delay (Minutes)')

sns.histplot(arr_subset['Arrival Delay in Minutes'], bins=30, kde=True, ax=axes[1], color='#ef4444')
axes[1].set_title('7B. Arrival Delay Distribution (1 - 180 mins)')
axes[1].set_xlabel('Arrival Delay (Minutes)')

plt.tight_layout()
plt.show()


### Chart 8: Satisfaction by Gender
**Question Answered**: Does class distribution differ by gender?


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x='Gender', hue='satisfaction', palette=['#f43f5e', '#10b981'])
plt.title('8. Satisfaction by Gender (Grouped Bar Chart)')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(title='Satisfaction')
plt.show()


### Chart 9: Satisfaction by Customer Type
**Question Answered**: How does satisfaction vary by customer type?


In [ ]:
cust_sat = pd.crosstab(train_df['Customer Type'], train_df['satisfaction'], normalize='index') * 100
ax = cust_sat.plot(kind='bar', stacked=True, color=['#f43f5e', '#10b981'], figsize=(8, 5))
plt.title('9. Normalized Satisfaction by Customer Type (%)')
plt.xlabel('Customer Type')
plt.ylabel('Percentage (%)')
plt.legend(title='Satisfaction', loc='upper right')
plt.xticks(rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontweight='bold')
plt.show()


### Chart 10: Satisfaction by Travel Class
**Question Answered**: How does satisfaction vary by class?


In [ ]:
class_sat = pd.crosstab(train_df['Class'], train_df['satisfaction'], normalize='index') * 100
ax = class_sat.loc[['Business', 'Eco', 'Eco Plus']].plot(kind='bar', stacked=True, color=['#f43f5e', '#10b981'], figsize=(9, 5))
plt.title('10. Normalized Satisfaction by Travel Class (%)')
plt.xlabel('Travel Class')
plt.ylabel('Percentage (%)')
plt.legend(title='Satisfaction', loc='upper right')
plt.xticks(rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', label_type='center', color='white', fontweight='bold')
plt.show()


### Chart 11: Satisfaction by Travel Type
**Question Answered**: How does satisfaction vary by travel purpose?


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x='Type of Travel', hue='satisfaction', palette=['#f43f5e', '#10b981'])
plt.title('11. Satisfaction by Travel Purpose (Business vs Personal)')
plt.xlabel('Type of Travel')
plt.ylabel('Count')
plt.legend(title='Satisfaction')
plt.show()


### Chart 12: Age vs Satisfaction Box Plot
**Question Answered**: Do age distributions differ by target class?


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=train_df, x='satisfaction', y='Age', palette=['#f43f5e', '#10b981'])
plt.title('12. Age Distribution by Satisfaction Class (Box Plot)')
plt.xlabel('Satisfaction Target')
plt.ylabel('Passenger Age (Years)')
plt.show()


### Chart 13: Flight Distance vs Satisfaction Box Plot
**Question Answered**: Do flight distances differ by target class?


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=train_df, x='satisfaction', y='Flight Distance', palette=['#f43f5e', '#10b981'])
plt.title('13. Flight Distance by Satisfaction (Box Plot)')
plt.xlabel('Satisfaction Target')
plt.ylabel('Flight Distance (Miles)')
plt.show()


### Chart 14: Delays vs Satisfaction Chart
**Question Answered**: How do delays compare across target classes?


In [ ]:
delay_means = train_df.groupby('satisfaction')[['Departure Delay in Minutes', 'Arrival Delay in Minutes']].mean()

ax = delay_means.plot(kind='bar', figsize=(9, 5), color=['#f59e0b', '#ef4444'])
plt.title('14. Average Delay Duration by Satisfaction Category')
plt.xlabel('Satisfaction')
plt.ylabel('Average Delay (Minutes)')
plt.xticks(rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f min', label_type='edge', padding=3)
plt.legend(['Departure Delay', 'Arrival Delay'])
plt.show()


### Chart 15: Average Service Ratings Across All Service Attributes
**Question Answered**: Which services have higher or lower average ratings?


In [ ]:
service_cols = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness'
]

avg_ratings = train_df[service_cols].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 7))
ax = avg_ratings.plot(kind='barh', color='#0284c7')
plt.title('15. Overall Average Rating Across All Service Attributes (0-5 Scale)')
plt.xlabel('Average Rating')
plt.xlim(0, 5)
for p in ax.patches:
    ax.annotate(f'{p.get_width():.2f}', (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points')
plt.show()


### Chart 16: Service Ratings by Satisfaction Class
**Question Answered**: Which ratings differ most between target groups?


In [ ]:
sat_ratings = train_df.groupby('satisfaction')[service_cols].mean().T
sat_ratings['Gap'] = sat_ratings['satisfied'] - sat_ratings['neutral or dissatisfied']
sat_ratings = sat_ratings.sort_values(by='Gap', ascending=True)

ax = sat_ratings[['neutral or dissatisfied', 'satisfied']].plot(
    kind='barh', figsize=(11, 8), color=['#f43f5e', '#10b981']
)
plt.title('16. Service Ratings Comparison by Satisfaction Class')
plt.xlabel('Average Score (0 - 5)')
plt.xlim(0, 5)
plt.legend(['Neutral / Dissatisfied', 'Satisfied'])
plt.show()


### Chart 17: Service Rating Box Plots
**Question Answered**: How much variation exists in ratings?


In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=train_df[service_cols], orient='h', palette='Blues_r')
plt.title('17. Service Rating Distributions and Variation (Box Plots)')
plt.xlabel('Rating (0 - 5)')
plt.show()


### Chart 18: Numerical Correlation Heatmap
**Question Answered**: Which numerical variables are associated?


In [ ]:
numeric_corr_cols = [
    'Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes'
] + service_cols

corr_mat = train_df[numeric_corr_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='Blues', cbar=True, square=True)
plt.title('18. Pearson Correlation Heatmap for Numerical & Rating Variables')
plt.show()


### Chart 19: Class and Travel Type Comparison
**Question Answered**: How do these two travel attributes combine?


In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=train_df, x='Class', hue='Type of Travel', palette='Set2')
plt.title('19. Cabin Class vs Travel Purpose Combination')
plt.xlabel('Travel Class')
plt.ylabel('Number of Passengers')
plt.legend(title='Type of Travel')
plt.show()


### Chart 20: Ratings by Customer Type
**Question Answered**: How do service ratings vary by customer loyalty?


In [ ]:
cust_ratings = train_df.groupby('Customer Type')[service_cols].mean().T

ax = cust_ratings.plot(kind='barh', figsize=(11, 7), color=['#f59e0b', '#10b981'])
plt.title('20. Mean Service Ratings by Customer Loyalty Tier')
plt.xlabel('Mean Rating (0 - 5)')
plt.xlim(0, 5)
plt.legend(title='Customer Type')
plt.show()


## 4. Machine Learning Preprocessing & Modeling Pipeline

According to Section 7 & 9 of the Report:
- Numerical features: `SimpleImputer(strategy='median')` + `StandardScaler()`
- Categorical features: `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')`
- Model: `LogisticRegression(max_iter=1000, random_state=42)`
- Stratified validation split from `train.csv` (80/20)
- Held-out test set `test.csv` untouched until final evaluation


In [ ]:
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
numeric_cols = [
    'Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding',
    'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness',
    'Departure Delay in Minutes', 'Arrival Delay in Minutes'
]
target_col = 'satisfaction'

y_train_full = (train_df[target_col] == 'satisfied').astype(int)
X_train_full = train_df[numeric_cols + categorical_cols]

y_test = (test_df[target_col] == 'satisfied').astype(int)
X_test = test_df[numeric_cols + categorical_cols]

# Stratified Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

# Pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_cols),
    ('cat', cat_pipeline, categorical_cols)
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs'))
])

print('Training pipeline defined. Fitting on training subset...')
model_pipeline.fit(X_train, y_train)
print('Training completed.')


## 5. Model Evaluation Metrics (Validation & Final Test Partition)


In [ ]:
# Validation Set Metrics
y_val_pred = model_pipeline.predict(X_valid)
y_val_prob = model_pipeline.predict_proba(X_valid)[:, 1]

print('=== VALIDATION SET RESULTS ===')
print(f'Accuracy:  {accuracy_score(y_valid, y_val_pred):.4f}')
print(f'Precision: {precision_score(y_valid, y_val_pred):.4f}')
print(f'Recall:    {recall_score(y_valid, y_val_pred):.4f}')
print(f'F1-score:  {f1_score(y_valid, y_val_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_valid, y_val_prob):.4f}')

# Final Test Set Evaluation
print('\nFitting on full training set for final held-out test evaluation...')
model_pipeline.fit(X_train_full, y_train_full)

y_test_pred = model_pipeline.predict(X_test)
y_test_prob = model_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_test_pred)
prec = precision_score(y_test, y_test_pred)
rec = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_prob)

print('\n=== FINAL HELD-OUT TEST RESULTS (test.csv) ===')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-score:  {f1:.4f}')
print(f'ROC-AUC:   {auc:.4f}')
print('\nClassification Report:\n', classification_report(y_test, y_test_pred, target_names=['Neutral/Dissat', 'Satisfied']))


## 6. Model Performance Visualizations (Section 9.5 of Report)
Visualizing:
1. Confusion Matrix Heatmap
2. Metric Comparison Bar Chart (Accuracy, Precision, Recall, F1)
3. ROC Curve with AUC Value
4. Actual vs. Predicted Class Counts
5. Predicted Probability Distribution
6. Horizontal Feature Coefficient Importance Chart


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Neutral/Dissat (0)', 'Satisfied (1)'],
            yticklabels=['Neutral/Dissat (0)', 'Satisfied (1)'])
axes[0].set_title('M1. Test Confusion Matrix Heatmap')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('True Class')

# 2. Metric Comparison Bar Chart
metrics_dict = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
bars = axes[1].bar(metrics_dict.keys(), metrics_dict.values(), color=['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6'])
axes[1].set_title('M2. Model Performance Metrics Comparison')
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
for b in bars:
    axes[1].annotate(f'{b.get_height()*100:.2f}%', (b.get_x() + b.get_width() / 2., b.get_height()),
                    ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
axes[0].plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'Logistic Regression (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='#94a3b8', linestyle='--', label='Random Chance (AUC = 0.50)')
axes[0].set_title('M3. Receiver Operating Characteristic (ROC) Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# 4. Actual vs Predicted Class Counts
comparison_df = pd.DataFrame({
    'Actual': ['Neutral/Dissat' if y == 0 else 'Satisfied' for y in y_test],
    'Predicted': ['Neutral/Dissat' if y == 0 else 'Satisfied' for y in y_test_pred]
})
actual_counts = comparison_df['Actual'].value_counts()
pred_counts = comparison_df['Predicted'].value_counts()

count_plot_df = pd.DataFrame({'Actual': actual_counts, 'Predicted': pred_counts})
count_plot_df.plot(kind='bar', ax=axes[1], color=['#64748b', '#2563eb'])
axes[1].set_title('M4. Actual vs. Predicted Class Counts')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Passenger Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 5. Predicted Probability Distribution
sns.histplot(y_test_prob[y_test == 1], color='#10b981', label='Actual: Satisfied', kde=True, ax=axes[0], bins=30)
sns.histplot(y_test_prob[y_test == 0], color='#f43f5e', label='Actual: Neutral/Dissat', kde=True, ax=axes[0], bins=30)
axes[0].set_title('M5. Predicted Probability Distribution by True Class')
axes[0].set_xlabel('Predicted Probability of Satisfaction')
axes[0].set_ylabel('Count')
axes[0].legend()

# 6. Top Logistic Regression Feature Coefficients
encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = encoder.get_feature_names_out(categorical_cols).tolist()
all_features = numeric_cols + cat_feature_names
coefs = model_pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({'Feature': all_features, 'Coefficient': coefs})
coef_df['Abs_Impact'] = coef_df['Coefficient'].abs()
top_coefs = coef_df.sort_values(by='Abs_Impact', ascending=False).head(15).sort_values(by='Coefficient', ascending=True)

colors = ['#f43f5e' if c < 0 else '#10b981' for c in top_coefs['Coefficient']]
axes[1].barh(top_coefs['Feature'], top_coefs['Coefficient'], color=colors)
axes[1].set_title('M6. Top 15 Logistic Regression Feature Coefficients')
axes[1].set_xlabel('Coefficient Value (Impact on Log-Odds)')
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()


## 7. Model Serialization & Export
Exporting the fitted pipeline to `backend/model/airline_satisfaction_model.pkl` for FastAPI serving.


In [ ]:
os.makedirs(os.path.join('..', 'backend', 'model'), exist_ok=True)
save_path = os.path.join('..', 'backend', 'model', 'airline_satisfaction_model.pkl')
if not os.path.exists(os.path.dirname(save_path)):
    save_path = os.path.join('backend', 'model', 'airline_satisfaction_model.pkl')

joblib.dump(model_pipeline, save_path)
print(f'Fitted preprocessing and model pipeline successfully exported to: {save_path}')
